In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import pandas as pd
import re


def parse_filename(path):
    name = path.stem

    # std_weighted:
    # fedavg_std_weighted_3_p1
    # fedavg_std_weighted_3_m1
    # fedavg_std_weighted_3_0
    match = re.match(
        r"(.+)_std_weighted_(\d+)_(p1|m1|0)$",
        name
    )

    if match:
        aggregator, top_k, sign_str = match.groups()

        sign_map = {
            "p1": 1,
            "m1": -1,
            "0": 0,
        }

        return {
            "aggregator": aggregator,
            "method": "std_weighted",
            "top_k": int(top_k),
            "sign": sign_map[sign_str],
        }

    # Single PC:
    # fedavg_pc1_p1
    # fedavg_pc1_m1
    # fedavg_pc1_0
    match = re.match(
        r"(.+)_pc(\d+)_(p1|m1|0)$",
        name
    )

    if match:
        aggregator, pc, sign_str = match.groups()

        sign_map = {
            "p1": 1,
            "m1": -1,
            "0": 0,
        }

        return {
            "aggregator": aggregator,
            "method": "pc",
            "top_k": int(pc),
            "sign": sign_map[sign_str],
        }

    return None

In [3]:
def load_experiments(result_dir):
    experiments = []

    for path in Path(result_dir).glob("*.csv"):

        config = parse_filename(path)

        if config is None:
            continue

        df = pd.read_csv(path)

        df["round"] = pd.to_numeric(
            df["round"],
            errors="coerce"
        )

        df["test_accuracy"] = pd.to_numeric(
            df["test_accuracy"],
            errors="coerce"
        )

        df = df.dropna(
            subset=["round", "test_accuracy"]
        )

        df = df.sort_values("round")

        config["path"] = path
        config["data"] = df

        experiments.append(config)

    return experiments

In [4]:
experiments = load_experiments("../results/pca_sign")

for exp in experiments:
    print(
        exp["aggregator"],
        exp["method"],
        exp["top_k"],
        exp["sign"]
    )

fedavg pc 2 1
fedavg pc 3 0
fedavg pc 3 -1
median pc 1 -1
trimmed_mean pc 2 0
fedavg pc 1 0
prodigy pc 1 1
multikrum pc 3 0
median pc 3 -1
fedavg pc 1 -1
krum pc 1 0
prodigy pc 3 1
prodigy pc 2 -1
krum pc 3 0
median pc 2 1
multikrum pc 1 0
trimmed_mean pc 3 0
krum pc 2 -1
multikrum pc 2 1
trimmed_mean pc 1 -1
multikrum pc 3 -1
fedavg pc 2 0
trimmed_mean pc 1 0
krum pc 3 1
krum pc 2 0
trimmed_mean pc 3 -1
multikrum pc 1 -1
krum pc 1 1
multikrum pc 2 0
trimmed_mean pc 2 1
prodigy pc 2 0
fedavg pc 2 -1
median pc 2 0
fedavg pc 3 1
median pc 1 1
prodigy pc 1 -1
median pc 3 1
fedavg pc 1 1
prodigy pc 3 -1
prodigy pc 2 1
median pc 2 -1
krum pc 2 1
multikrum pc 2 -1
median pc 1 0
prodigy pc 1 0
trimmed_mean pc 1 1
prodigy pc 3 0
median pc 3 0
multikrum pc 3 1
krum pc 3 -1
trimmed_mean pc 3 1
multikrum pc 1 1
krum pc 1 -1
trimmed_mean pc 2 -1


In [5]:
import numpy as np
import pandas as pd


def summarize_experiments(experiments):
    rows = []

    for exp in experiments:
        df = exp["data"]

        final_acc = df["test_accuracy"].iloc[-1]
        last5_acc = df["test_accuracy"].tail(5).mean()

        auc = np.trapezoid(
            df["test_accuracy"],
            df["round"]
        )

        round_range = (
            df["round"].max()
            - df["round"].min()
        )

        normalized_auc = auc / round_range

        rows.append({
            "aggregator": exp["aggregator"],
            "method": exp["method"],
            "top_k": exp["top_k"],
            "sign": exp["sign"],
            "final_acc": final_acc,
            "last5_acc": last5_acc,
            "auc": normalized_auc,
            "file": exp["path"].name,
        })

    return pd.DataFrame(rows)

In [6]:
summary = summarize_experiments(experiments)

summary = summary.sort_values("last5_acc")

print(summary.to_string(index=False))

  aggregator method  top_k  sign  final_acc  last5_acc      auc                    file
trimmed_mean     pc      2    -1     0.7713    0.72966 0.585138 trimmed_mean_pc2_m1.csv
trimmed_mean     pc      3     1     0.7814    0.74258 0.556335 trimmed_mean_pc3_p1.csv
        krum     pc      2     0     0.6789    0.74886 0.615098          krum_pc2_0.csv
   multikrum     pc      3     1     0.7331    0.75508 0.534008    multikrum_pc3_p1.csv
   multikrum     pc      3    -1     0.9224    0.75758 0.510630    multikrum_pc3_m1.csv
      median     pc      2    -1     0.7405    0.76644 0.543667       median_pc2_m1.csv
   multikrum     pc      2     0     0.7389    0.76978 0.551338     multikrum_pc2_0.csv
      median     pc      2     0     0.8386    0.78010 0.578895        median_pc2_0.csv
        krum     pc      3    -1     0.7750    0.78066 0.541575         krum_pc3_m1.csv
     prodigy     pc      1     1     0.5958    0.78802 0.612420      prodigy_pc1_p1.csv
trimmed_mean     pc      2     1

In [7]:
best_per_aggregator = (
    summary
    .sort_values("last5_acc")
    .groupby("aggregator")
    .first()
    .reset_index()
)

print(best_per_aggregator)

     aggregator method  top_k  sign  final_acc  last5_acc       auc  \
0        fedavg     pc      2     1     0.7690    0.79656  0.564225   
1          krum     pc      2     0     0.6789    0.74886  0.615098   
2        median     pc      2    -1     0.7405    0.76644  0.543667   
3     multikrum     pc      3     1     0.7331    0.75508  0.534008   
4       prodigy     pc      1     1     0.5958    0.78802  0.612420   
5  trimmed_mean     pc      2    -1     0.7713    0.72966  0.585138   

                      file  
0        fedavg_pc2_p1.csv  
1           krum_pc2_0.csv  
2        median_pc2_m1.csv  
3     multikrum_pc3_p1.csv  
4       prodigy_pc1_p1.csv  
5  trimmed_mean_pc2_m1.csv  


In [10]:
sign_summary = (
    summary
    .groupby(["top_k", "sign"])
    ["last5_acc"]
    .mean()
    .reset_index()
    .sort_values("last5_acc")
)

print(sign_summary)

   top_k  sign  last5_acc
3      2    -1   0.795243
4      2     0   0.805607
8      3     1   0.807517
6      3    -1   0.814270
5      2     1   0.831917
2      1     1   0.838120
7      3     0   0.843763
0      1    -1   0.855880
1      1     0   0.867680
